# 12 — Distillation Evaluation: Student vs Teacher

Evaluate how well the distilled ONNX students reproduce the GPT-4o teacher's scores.

## Metrics
- **MAE** < 10 per signal (student vs teacher agreement)
- **Pearson r** > 0.7 per signal
- **Consistency**: same image 5x → variance (student should be MORE consistent than teacher)
- **Latency**: ONNX should be 100-1000x faster than GPT-4o
- **Cross-signal correlation**: student should preserve teacher's inter-signal relationships
- **Age-elasticity check**: UTKFace age labels validate elasticity signal makes biological sense

In [ ]:
!pip install -q onnxruntime opencv-python scikit-image scikit-learn matplotlib seaborn scipy pandas numpy tqdm

In [ ]:
import json
import time
import numpy as np
import cv2
import onnxruntime as ort
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Glowlytics')
except Exception:
    BASE_DIR = Path('./data')

LABELS_DIR = BASE_DIR / 'datasets' / 'teacher_labels'
MODELS_DIR = BASE_DIR / 'models' / 'distilled' / 'onnx'
RESULTS_DIR = BASE_DIR / 'results' / 'distillation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SIGNALS = ['structure', 'hydration', 'inflammation', 'sunDamage', 'elasticity']
SIGNAL_COLORS = {
    'structure': '#7DE7E1', 'hydration': '#4DA6FF', 'inflammation': '#FF7A78',
    'sunDamage': '#F2B56A', 'elasticity': '#B68AFF',
}
TARGET_MAE = 10.0
TARGET_R = 0.7

# ImageNet normalization
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

## 1. Load Models & Test Data

In [ ]:
# Reuse feature extraction from notebook 11
from skimage.feature import local_binary_pattern
from skimage.filters import frangi


def build_gabor_bank():
    kernels = []
    for theta_idx in range(4):
        theta = theta_idx * np.pi / 4
        for freq in [0.1, 0.25, 0.4]:
            kernel = cv2.getGaborKernel(ksize=(31, 31), sigma=4.0, theta=theta,
                                        lambd=1.0 / freq, gamma=0.5, psi=0)
            kernels.append(kernel)
    return kernels

GABOR_BANK = build_gabor_bank()


def extract_hydration_features(gray, image):
    gabor = []
    for k in GABOR_BANK:
        r = cv2.filter2D(gray, cv2.CV_64F, k)
        gabor.extend([r.mean(), r.std()])
    lbp = local_binary_pattern(gray, 16, 2, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=18, range=(0, 18), density=True)
    gray_only = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if image.ndim == 3 else image
    hl = (gray_only > 220).astype(np.float32)
    ratio = hl.mean()
    spread = np.argwhere(hl > 0).std(axis=0).mean() / max(gray.shape) if hl.sum() > 0 else 0
    return np.array(gabor + list(lbp_hist) + [ratio, spread], dtype=np.float32)


def extract_elasticity_features(gray):
    h, w = gray.shape
    rois = [gray[0:h//4, w//4:3*w//4], gray[h//4:h//2, 0:w//3], gray[h//4:h//2, 2*w//3:w]]
    feats = []
    for roi in rois:
        if roi.size < 100:
            feats.extend([0, 0, 0])
            continue
        r = frangi(roi.astype(np.float64)/255, sigmas=range(1,5), alpha=0.5, beta=0.5, gamma=15, black_ridges=True)
        d = float((r > 0.01).mean())
        i = float(r[r > 0.01].mean()) if d > 0 else 0
        feats.extend([d, i, float(r.max())])
    # Landmark proxy
    h, w = gray.shape
    geom = [gray[0:h//5, w//4:3*w//4].mean()/255, gray[h//5:2*h//5, w//8:3*w//8].mean()/255,
            gray[h//5:2*h//5, 5*w//8:7*w//8].mean()/255, gray[2*h//5:3*h//5, 3*w//8:5*w//8].mean()/255,
            gray[4*h//5:h, w//4:3*w//4].mean()/255]
    return np.array(feats + geom, dtype=np.float32)


def prepare_image(path, size=224):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size))
    return img


def normalize_image(img):
    """ImageNet normalize and convert to NCHW float32."""
    x = img.astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    return x.transpose(2, 0, 1)[np.newaxis]  # [1, 3, H, W]

In [ ]:
# Load test data (teacher labels)
test_path = LABELS_DIR / 'test.json'
if test_path.exists():
    with open(test_path) as f:
        test_data = json.load(f)
    # Filter to existing images
    test_data = [t for t in test_data if Path(t['image_path']).exists()]
    print(f'Test set: {len(test_data)} images')
else:
    print('No test data found. Run notebooks 10 and 11 first.')
    test_data = []

# Load ONNX models
sessions = {}
for name in ['structure', 'hydration', 'elasticity']:
    p = MODELS_DIR / f'{name}.onnx'
    if p.exists():
        sessions[name] = ort.InferenceSession(str(p))
        print(f'Loaded {name} model')
    else:
        print(f'{name} model not found at {p}')

multi_path = MODELS_DIR / 'multi_signal.onnx'
if multi_path.exists():
    sessions['multi'] = ort.InferenceSession(str(multi_path))
    print('Loaded multi-signal model')

## 2. Run Student Inference on Test Set

In [ ]:
def run_student_inference(test_data, sessions):
    """Run all student models on the test set. Returns predictions dict."""
    results = {sig: {'teacher': [], 'student': [], 'latency_ms': []} for sig in SIGNALS}

    for entry in tqdm(test_data, desc='Student inference'):
        img = prepare_image(entry['image_path'])
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        img_tensor = normalize_image(img)
        teacher = entry['signals']

        # Structure
        if 'structure' in sessions:
            start = time.perf_counter()
            out = sessions['structure'].run(None, {'image': img_tensor})
            elapsed = (time.perf_counter() - start) * 1000
            score = float(np.clip(out[0][0][-1], 0, 100))  # Last output = structure_score
            results['structure']['student'].append(score)
            results['structure']['teacher'].append(teacher['structure'])
            results['structure']['latency_ms'].append(elapsed)

        # Hydration
        if 'hydration' in sessions:
            hc = extract_hydration_features(gray, img)[np.newaxis]
            start = time.perf_counter()
            out = sessions['hydration'].run(None, {'image': img_tensor, 'handcrafted_features': hc})
            elapsed = (time.perf_counter() - start) * 1000
            score = float(np.clip(out[0][0][0], 0, 100))
            results['hydration']['student'].append(score)
            results['hydration']['teacher'].append(teacher['hydration'])
            results['hydration']['latency_ms'].append(elapsed)

        # Elasticity
        if 'elasticity' in sessions:
            hc = extract_elasticity_features(gray)[np.newaxis]
            start = time.perf_counter()
            out = sessions['elasticity'].run(None, {'image': img_tensor, 'handcrafted_features': hc})
            elapsed = (time.perf_counter() - start) * 1000
            score = float(np.clip(out[0][0][0], 0, 100))
            results['elasticity']['student'].append(score)
            results['elasticity']['teacher'].append(teacher['elasticity'])
            results['elasticity']['latency_ms'].append(elapsed)

        # Multi-signal (for inflammation and sunDamage, which lack dedicated models)
        if 'multi' in sessions:
            all_hc = np.concatenate([
                extract_hydration_features(gray, img),
                extract_elasticity_features(gray),
            ])[np.newaxis]
            start = time.perf_counter()
            out = sessions['multi'].run(None, {'image': img_tensor, 'handcrafted_features': all_hc})
            elapsed = (time.perf_counter() - start) * 1000
            multi_scores = out[0][0]  # [structure, hydration, inflammation, sunDamage, elasticity]

            for i, sig in enumerate(SIGNALS):
                score = float(np.clip(multi_scores[i], 0, 100))
                # Use multi-signal model for signals that lack a dedicated model
                if sig not in sessions:
                    results[sig]['student'].append(score)
                    results[sig]['teacher'].append(teacher[sig])
                    results[sig]['latency_ms'].append(elapsed / 5)  # Amortized

    return results


if test_data and sessions:
    inference_results = run_student_inference(test_data, sessions)
else:
    print('Cannot run inference without test data and models.')
    inference_results = None

## 3. Per-Signal Metrics

In [ ]:
def compute_metrics(results):
    """Compute per-signal agreement metrics."""
    report = {}

    for sig in SIGNALS:
        t = np.array(results[sig]['teacher'])
        s = np.array(results[sig]['student'])
        lat = results[sig]['latency_ms']

        if len(t) < 2:
            report[sig] = {'n': 0, 'status': 'NO DATA'}
            continue

        mae = float(np.abs(s - t).mean())
        rmse = float(np.sqrt(((s - t) ** 2).mean()))
        r, p_val = stats.pearsonr(t, s)
        rho, _ = stats.spearmanr(t, s)
        median_lat = float(np.median(lat))

        passes_mae = mae < TARGET_MAE
        passes_r = r > TARGET_R

        report[sig] = {
            'n': len(t),
            'mae': round(mae, 2),
            'rmse': round(rmse, 2),
            'pearson_r': round(r, 4),
            'pearson_p': round(p_val, 6),
            'spearman_rho': round(rho, 4),
            'median_latency_ms': round(median_lat, 1),
            'passes_mae': passes_mae,
            'passes_r': passes_r,
            'status': 'PASS' if (passes_mae and passes_r) else 'FAIL',
        }

    return report


if inference_results:
    metrics = compute_metrics(inference_results)

    print(f'\n{"="*80}')
    print(f'{"Signal":<16} {"N":>6} {"MAE":>8} {"RMSE":>8} {"r":>8} {"rho":>8} {"ms":>8} {"Status":>8}')
    print(f'{"-"*80}')
    passing = 0
    for sig in SIGNALS:
        m = metrics[sig]
        if m.get('n', 0) > 0:
            status = m['status']
            if status == 'PASS':
                passing += 1
            print(f'{sig:<16} {m["n"]:>6} {m["mae"]:>8.2f} {m["rmse"]:>8.2f} '
                  f'{m["pearson_r"]:>8.4f} {m["spearman_rho"]:>8.4f} '
                  f'{m["median_latency_ms"]:>8.1f} {status:>8}')
    print(f'{"="*80}')
    print(f'Overall: {passing}/{len(SIGNALS)} signals passing')

## 4. Scatter Plots (Predicted vs Teacher)

In [ ]:
def plot_scatter_all(results, metrics):
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))

    for ax, sig in zip(axes, SIGNALS):
        t = np.array(results[sig]['teacher'])
        s = np.array(results[sig]['student'])
        m = metrics[sig]

        if len(t) < 2:
            ax.set_title(f'{sig}\nNO DATA')
            continue

        ax.scatter(t, s, alpha=0.3, s=8, c=SIGNAL_COLORS[sig])
        ax.plot([0, 100], [0, 100], 'k--', alpha=0.3, label='Perfect')

        # Regression line
        z = np.polyfit(t, s, 1)
        p = np.poly1d(z)
        ax.plot([0, 100], [p(0), p(100)], 'r--', alpha=0.7, label='Fit')

        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_aspect('equal')
        ax.set_xlabel('Teacher Score')
        ax.set_ylabel('Student Score')
        ax.set_title(f'{sig}\nMAE={m["mae"]:.1f}, r={m["pearson_r"]:.3f} [{m["status"]}]')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.2)

    plt.suptitle('Student vs Teacher Agreement', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'scatter_student_vs_teacher.png'), dpi=150, bbox_inches='tight')
    plt.show()


if inference_results:
    plot_scatter_all(inference_results, metrics)

## 5. Error Distribution

In [ ]:
def plot_error_distributions(results):
    fig, axes = plt.subplots(1, 5, figsize=(25, 4))

    for ax, sig in zip(axes, SIGNALS):
        t = np.array(results[sig]['teacher'])
        s = np.array(results[sig]['student'])
        if len(t) < 2:
            continue
        errors = s - t
        ax.hist(errors, bins=40, color=SIGNAL_COLORS[sig], alpha=0.8, edgecolor='white')
        ax.axvline(0, color='red', linestyle='--', alpha=0.5)
        ax.axvline(errors.mean(), color='black', linestyle='-', alpha=0.7,
                   label=f'bias={errors.mean():.1f}')
        ax.set_xlabel('Error (student - teacher)')
        ax.set_ylabel('Count')
        ax.set_title(sig)
        ax.legend(fontsize=8)

    plt.suptitle('Prediction Error Distributions', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'error_distributions.png'), dpi=150, bbox_inches='tight')
    plt.show()


if inference_results:
    plot_error_distributions(inference_results)

## 6. Cross-Signal Correlation Preservation

The student should preserve the teacher's inter-signal correlation structure.
e.g., if teacher says inflammation and structure are negatively correlated,
the student should too.

In [ ]:
def plot_correlation_comparison(results):
    teacher_df = pd.DataFrame({sig: results[sig]['teacher'] for sig in SIGNALS
                               if len(results[sig]['teacher']) > 0})
    student_df = pd.DataFrame({sig: results[sig]['student'] for sig in SIGNALS
                               if len(results[sig]['student']) > 0})

    if teacher_df.empty or student_df.empty:
        print('Not enough data for correlation comparison.')
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Teacher correlations
    t_corr = teacher_df.corr()
    sns.heatmap(t_corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1,
                ax=axes[0], square=True)
    axes[0].set_title('Teacher Correlations')

    # Student correlations
    s_corr = student_df.corr()
    sns.heatmap(s_corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1,
                ax=axes[1], square=True)
    axes[1].set_title('Student Correlations')

    # Difference
    common_cols = list(set(t_corr.columns) & set(s_corr.columns))
    diff = s_corr.loc[common_cols, common_cols] - t_corr.loc[common_cols, common_cols]
    sns.heatmap(diff, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-0.5, vmax=0.5,
                ax=axes[2], square=True)
    axes[2].set_title('Difference (Student - Teacher)')

    plt.suptitle('Cross-Signal Correlation Preservation', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'correlation_preservation.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Frobenius norm of difference
    frob = float(np.sqrt((diff.values ** 2).sum()))
    print(f'Correlation matrix Frobenius distance: {frob:.3f} (lower is better)')


if inference_results:
    plot_correlation_comparison(inference_results)

## 7. Age-Elasticity Biological Validation

Using UTKFace images (which have age labels), verify that the student's
elasticity predictions decrease with age — a basic sanity check.

In [ ]:
def age_elasticity_validation(test_data, results):
    """Check student elasticity scores vs known age labels."""
    age_data = []
    for i, entry in enumerate(test_data):
        age = entry.get('age')
        if age is not None and i < len(results['elasticity']['student']):
            age_data.append({
                'age': age,
                'teacher_elasticity': results['elasticity']['teacher'][i],
                'student_elasticity': results['elasticity']['student'][i],
            })

    if len(age_data) < 20:
        print(f'Only {len(age_data)} age-labeled samples — insufficient for validation.')
        return

    df = pd.DataFrame(age_data)
    print(f'Age-labeled samples: {len(df)}')

    # Correlation
    r_teacher, _ = stats.pearsonr(df['age'], df['teacher_elasticity'])
    r_student, _ = stats.pearsonr(df['age'], df['student_elasticity'])
    print(f'Age-elasticity correlation:')
    print(f'  Teacher: r={r_teacher:.3f} (expect negative)')
    print(f'  Student: r={r_student:.3f} (expect negative)')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, col, label, color in [
        (axes[0], 'teacher_elasticity', 'Teacher', '#B68AFF'),
        (axes[1], 'student_elasticity', 'Student', '#7DE7E1'),
    ]:
        ax.scatter(df['age'], df[col], alpha=0.3, s=10, c=color)
        z = np.polyfit(df['age'], df[col], 2)
        p = np.poly1d(z)
        x = np.linspace(df['age'].min(), df['age'].max(), 100)
        ax.plot(x, p(x), 'r--', linewidth=2)
        r, _ = stats.pearsonr(df['age'], df[col])
        ax.set_xlabel('Age')
        ax.set_ylabel('Elasticity Score')
        ax.set_title(f'{label} (r={r:.3f})')
        ax.grid(True, alpha=0.2)

    plt.suptitle('Age vs Elasticity — Biological Validation', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'age_elasticity_validation.png'), dpi=150, bbox_inches='tight')
    plt.show()


if inference_results and test_data:
    age_elasticity_validation(test_data, inference_results)

## 8. Latency Comparison

In [ ]:
def plot_latency(results):
    fig, ax = plt.subplots(figsize=(10, 5))

    latencies = {}
    for sig in SIGNALS:
        lat = results[sig]['latency_ms']
        if lat:
            latencies[sig] = np.median(lat)

    if not latencies:
        print('No latency data.')
        return

    # Student latencies
    signals = list(latencies.keys())
    student_ms = [latencies[s] for s in signals]

    # GPT-4o baseline (~4000ms)
    teacher_ms = [4000] * len(signals)

    x = np.arange(len(signals))
    width = 0.35
    bars1 = ax.bar(x - width/2, teacher_ms, width, label='GPT-4o Teacher',
                   color='#FF7A78', alpha=0.7)
    bars2 = ax.bar(x + width/2, student_ms, width, label='ONNX Student',
                   color='#7DE7E1', alpha=0.7)

    ax.set_ylabel('Latency (ms)')
    ax.set_yscale('log')
    ax.set_xticks(x)
    ax.set_xticklabels(signals)
    ax.legend()
    ax.set_title('Inference Latency: Teacher vs Student')

    # Add speedup annotations
    for i, (t, s) in enumerate(zip(teacher_ms, student_ms)):
        speedup = t / s if s > 0 else 0
        ax.text(i + width/2, s * 1.5, f'{speedup:.0f}x', ha='center', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'latency_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()


if inference_results:
    plot_latency(inference_results)

## 9. Generate Final Report

In [ ]:
def generate_final_report(metrics, results_dir):
    """Generate JSON and markdown evaluation report."""
    report = {
        'pipeline': 'GPT-4o Teacher → ONNX Student Distillation',
        'thresholds': {'mae': TARGET_MAE, 'pearson_r': TARGET_R},
        'signals': metrics,
        'summary': {
            'total_signals': len(SIGNALS),
            'passing': sum(1 for m in metrics.values() if m.get('status') == 'PASS'),
            'avg_mae': round(np.mean([m['mae'] for m in metrics.values() if 'mae' in m]), 2),
            'avg_r': round(np.mean([m['pearson_r'] for m in metrics.values() if 'pearson_r' in m]), 4),
        },
    }

    # JSON
    with open(results_dir / 'distillation_report.json', 'w') as f:
        json.dump(report, f, indent=2)

    # Markdown
    md = '# Distillation Evaluation Report\n\n'
    md += f'Pipeline: GPT-4o Teacher → ONNX Student\n\n'
    md += '## Per-Signal Results\n\n'
    md += '| Signal | N | MAE | RMSE | Pearson r | Latency (ms) | Status |\n'
    md += '|--------|---|-----|------|-----------|-------------|--------|\n'
    for sig in SIGNALS:
        m = metrics[sig]
        if m.get('n', 0) > 0:
            md += (f'| {sig} | {m["n"]} | {m["mae"]} | {m["rmse"]} | '
                   f'{m["pearson_r"]} | {m["median_latency_ms"]} | {m["status"]} |\n')
    s = report['summary']
    md += f'\n## Summary\n\n'
    md += f'- **Passing**: {s["passing"]}/{s["total_signals"]} signals\n'
    md += f'- **Avg MAE**: {s["avg_mae"]} (target < {TARGET_MAE})\n'
    md += f'- **Avg Pearson r**: {s["avg_r"]} (target > {TARGET_R})\n'

    with open(results_dir / 'distillation_report.md', 'w') as f:
        f.write(md)

    print(md)
    return report


if inference_results:
    final_report = generate_final_report(metrics, RESULTS_DIR)
    print(f'\nReports saved to {RESULTS_DIR}')